# 1. What are Missing Values?

### Concept & Definition
Missing values occur when no data value is stored for an attribute in an observation. In Python, missing values are typically represented as `NaN` (Not a Number), `None`, `null`, or explicit missing indicators (e.g., `"-999"`, `"N/A"`).

### Real-World / Business Example
In a customer survey, optional questions like `Annual Income` or `Middle Name` are frequently left empty by respondents.

### ML Impact
Most scikit-learn algorithms (e.g., Linear Regression, SVMs, Logistic Regression, Decision Trees in older implementations) throw errors when encountering `NaN` values during matrix calculations.

In [1]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Customer_Data.csv")

# Ensure numeric parsing for MonthlyCharges and Age for accurate missing value checks
df["MonthlyCharges"] = pd.to_numeric(
    df["MonthlyCharges"].astype(str).str.replace("$", ""), errors="coerce"
)
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

print("=== Total Missing Values per Column ===")
print(df.isnull().sum())

=== Total Missing Values per Column ===
CustomerID         0
Age               54
Gender            48
TenureYears        0
MonthlyCharges    99
TotalCharges       0
ContractType       0
PaymentMethod      0
Churn              0
dtype: int64


# 2. Why Missing Values Occur

### Concept & Definition
Missing data occurs due to system failures, human error, incomplete responses, data extraction corruptions, or intentional non-disclosure. Understanding *why* data is missing helps guide the selection of an appropriate imputation strategy.

### Real-World / Business Example
- **Sensor Glitch:** A temperature sensor momentarily loses internet connectivity and drops data packets.
- **Privacy Choice:** A user deliberately skips entering their annual income on an online application form.

### ML Impact
Imputing missing data without understanding the root cause introduces severe bias or distorts the true variance of the dataset.

In [2]:
# Inspecting pattern of missingness across Age and MonthlyCharges
missing_summary = pd.DataFrame({
    "Null Count": df.isnull().sum(),
    "Null Percentage (%)": (df.isnull().sum() / len(df)) * 100
})
display(missing_summary[missing_summary["Null Count"] > 0])

,Null Count,Null Percentage (%)
Age,54,5.346535
Gender,48,4.752475
MonthlyCharges,99,9.801980


# 3. Missing Data Mechanisms: MCAR, MAR, & MNAR

### Concepts & Definitions
1. **Missing Completely at Random (MCAR):** The probability of a value being missing is completely random and entirely unrelated to any observed or unobserved variable (e.g., a coin flip dropped a record).
2. **Missing at Random (MAR):** Missingness is systematic, but dependent strictly on *other observed variables* (e.g., older customers in a survey are less likely to report `MonthlyCharges`).
3. **Missing Not at Random (MNAR):** Missingness depends directly on the unobserved value itself (e.g., people with extremely high incomes refuse to disclose their `Annual Income`).

### Imputation Rule of Thumb
- **MCAR:** Mean/Median imputation or dropping rows is generally safe.
- **MAR:** Model-based or group-based imputation (KNN, Iterative Imputer) works best.
- **MNAR:** Simple imputation will introduce bias; flag missing values using indicator variables (e.g., `Missing_Indicator=1`).

In [3]:
# Programmatically checking if Age missingness depends on Gender (MAR test sample)
df_mar_check = df.copy()
df_mar_check["Age_Missing"] = df_mar_check["Age"].isnull()

print("=== Proportion of Missing Age across Gender Categories (MAR Assessment) ===")
print(pd.crosstab(df_mar_check["Gender"], df_mar_check["Age_Missing"], normalize="index") * 100)

=== Proportion of Missing Age across Gender Categories (MAR Assessment) ===
Age_Missing      False     True 
Gender                          
F            92.187500  7.812500
Female       94.471154  5.528846
M            95.789474  4.210526
Male         94.832041  5.167959


# 4. Detecting Missing Values

### Concept & Definition
Detecting missing values involves identifying both explicit `NaN`/`None` entries and implicit markers like `"?"`, `"N/A"`, `"-999"`, or empty spaces.

### Real-World / Business Example
A legacy database flags missing values as `"-999"` instead of SQL `NULL`. Standard `.isnull()` calls miss these unless they are replaced first.

### ML Impact
Unflagged implicit null values act as false valid extreme values during scaling and model fitting.

In [4]:
# Detecting explicit and implicit missing values
df_detect = df.copy()

# Convert potential implicit missing markers to true NaN
df_detect.replace(["?", "N/A", "INVALID", "-999", "Unknown", " "], np.nan, inplace=True)

print("=== Comprehensive Null Count Post Implicit Cleanup ===")
print(df_detect.isnull().sum())

=== Comprehensive Null Count Post Implicit Cleanup ===
CustomerID          0
Age                54
Gender             48
TenureYears         0
MonthlyCharges     99
TotalCharges        0
ContractType      171
PaymentMethod       0
Churn               0
dtype: int64


# 5. Missing Value Percentage

### Concept & Definition
Calculates the proportion of null entries per column to determine whether to drop features, drop rows, or perform advanced imputation.

### Business Decision Thresholds:
- **< 5% Missing:** Drop rows or simple mean/median/mode imputation.
- **5% – 30% Missing:** Model-based imputation (KNN / Group Imputation).
- **> 40%–50% Missing:** Drop column unless the feature is a critical business driver.

In [5]:
# Calculate Missing Percentage
null_pct = (df_detect.isnull().sum() / len(df_detect)) * 100
null_pct_df = pd.DataFrame({"Missing Percentage (%)": null_pct.round(2)})

print("=== Feature Missing Percentages ===")
display(null_pct_df.sort_values(by="Missing Percentage (%)", ascending=False))

=== Feature Missing Percentages ===


,Missing Percentage (%)
ContractType,16.93
MonthlyCharges,9.80
Age,5.35
Gender,4.75
CustomerID,0.00
TenureYears,0.00
TotalCharges,0.00
PaymentMethod,0.00
Churn,0.00


# 6. Dropping Rows

### Concept & Definition
Removing entire rows/observations that contain missing values in specific columns using `dropna(subset=[...])`.

### When to Use:
- When missingness is confirmed **MCAR**.
- When missing values account for **< 5%** of total dataset records.

### When NOT to Use:
- When the dataset size is small.
- When missingness is systematic (MAR/MNAR).

### Advantages & Limitations:
- **Advantage:** Preserves true distributions without adding synthetic values.
- **Limitation:** Loses statistical power and can introduce bias if data is not MCAR.

In [6]:
# Dropping rows where Age is missing
df_drop_rows = df_detect.copy()
print(f"Shape Before Dropping Rows: {df_drop_rows.shape}")

df_drop_rows = df_drop_rows.dropna(subset=["Age"])
print(f"Shape After Dropping Missing Age Rows: {df_drop_rows.shape}")

Shape Before Dropping Rows: (1010, 9)
Shape After Dropping Missing Age Rows: (956, 9)


# 7. Dropping Columns

### Concept & Definition
Completely deleting features/columns from the dataset using `drop(columns=[...])`.

### When to Use:
- When a column is missing **> 50%** of its values and cannot be reliably reconstructed.
- When the feature is non-critical for prediction.

### When NOT to Use:
- When a column with missing data has strong domain-specific predictive power.

### Advantages & Limitations:
- **Advantage:** Eliminates noisy, mostly empty features.
- **Limitation:** Total loss of information contained in non-null entries of that column.

In [7]:
# Dropping columns exceeding threshold (e.g., > 40% missing)
threshold = 40.0
cols_to_drop = null_pct_df[null_pct_df["Missing Percentage (%)"] > threshold].index.tolist()

df_drop_cols = df_detect.drop(columns=cols_to_drop)
print(f"Columns Dropped (> {threshold}% missing): {cols_to_drop}")
print(f"Remaining Shape: {df_drop_cols.shape}")

Columns Dropped (> 40.0% missing): []
Remaining Shape: (1010, 9)


# 8. Mean Imputation

### Concept & Definition
Replaces missing numerical values with the overall mathematical mean of that column.

### When to Use:
- Normally distributed numerical features with **no extreme outliers**.

### When NOT to Use:
- Skewed features or columns with extreme outliers.

### Advantages & Limitations:
- **Advantage:** Fast, simple, and preserves column mean.
- **Limitation:** Distorts variance, shrinks standard deviation, and distorts covariance with other variables.

In [8]:
# Mean Imputation on MonthlyCharges
df_mean = df_detect.copy()
mean_val = df_mean["MonthlyCharges"].mean()

df_mean["MonthlyCharges_Imputed"] = df_mean["MonthlyCharges"].fillna(mean_val)

print(f"Calculated Mean: {mean_val:.2f}")
print(f"Variance Before: {df_mean['MonthlyCharges'].var():.2f}")
print(f"Variance After Mean Imputation: {df_mean['MonthlyCharges_Imputed'].var():.2f}")

Calculated Mean: 64.88
Variance Before: 873.22
Variance After Mean Imputation: 787.55


# 9. Median Imputation

### Concept & Definition
Replaces missing numerical values with the 50th percentile (median) value of that feature.

### When to Use:
- Skewed numerical data or features containing heavy outliers (e.g., Income, Charges).

### When NOT to Use:
- Clean, perfectly normally distributed data (where mean and median are equal).

### Advantages & Limitations:
- **Advantage:** Robust against outliers; does not warp central tendency.
- **Limitation:** Still reduces feature variance.

In [9]:
# Median Imputation on Age
df_median = df_detect.copy()
median_val = df_median["Age"].median()

df_median["Age_Imputed"] = df_median["Age"].fillna(median_val)

print(f"Calculated Median Age: {median_val:.2f}")
print(f"Null Count After Imputation: {df_median['Age_Imputed'].isnull().sum()}")

Calculated Median Age: 45.00
Null Count After Imputation: 0


# 10. Mode Imputation

### Concept & Definition
Replaces missing values with the most frequently occurring category or discrete value in the feature.

### When to Use:
- Categorical features with low missingness (e.g., `Gender`, `PaymentMethod`).

### When NOT to Use:
- Uniformly distributed categorical variables without a clear dominant mode.

### Advantages & Limitations:
- **Advantage:** Preserves original string category choices.
- **Limitation:** Artificially inflates the frequency of the dominant mode category.

In [10]:
# Mode Imputation on Gender
df_mode = df_detect.copy()
mode_val = df_mode["Gender"].mode()[0]

df_mode["Gender_Imputed"] = df_mode["Gender"].fillna(mode_val)

print(f"Identified Most Frequent Mode: '{mode_val}'")
print(f"Null Count After Mode Imputation: {df_mode['Gender_Imputed'].isnull().sum()}")

Identified Most Frequent Mode: 'Female'
Null Count After Mode Imputation: 0


# 11. Constant Value Imputation

### Concept & Definition
Fills missing entries with a predefined static string or scalar value (e.g., `"Missing"`, `"Unknown"`, `-999`).

### When to Use:
- Categorical features where missingness represents an independent semantic state.

### When NOT to Use:
- Numerical algorithms that treat numeric placeholders (like `-999`) as true magnitude values.

### Advantages & Limitations:
- **Advantage:** Explicitly models missingness as an explicit category without altering known categories.
- **Limitation:** Adds a new category level, increasing cardinality.

In [11]:
# Constant Value Imputation for ContractType
df_const = df_detect.copy()
df_const["ContractType_Imputed"] = df_const["ContractType"].fillna("Unknown_Contract")

print("Value Counts After Constant Imputation:")
print(df_const["ContractType_Imputed"].value_counts())

Value Counts After Constant Imputation:
ContractType_Imputed
month to month      204
Unknown_Contract    171
Two year            167
Month-to-month      164
One year            158
m2m                 146
Name: count, dtype: int64


# 12. Forward Fill & Backward Fill

### Concept & Definition
- **Forward Fill (`ffill`):** Propagates the last known non-null observation forward.
- **Backward Fill (`bfill`):** Propagates the next known non-null observation backward.

### When to Use:
- Time-series, sequential, or ordered financial data.

### When NOT to Use:
- Cross-sectional, tabular datasets with randomly shuffled observations.

### Advantages & Limitations:
- **Advantage:** Preserves temporal trends and local continuity.
- **Limitation:** Fails if initial or final boundary values are null.

In [12]:
# Demonstrating Forward & Backward Fill on Sequential Data
df_seq = pd.DataFrame({"Time": [1, 2, 3, 4, 5], "Value": [10.5, np.nan, np.nan, 25.0, 30.0]})

df_seq["FFill"] = df_seq["Value"].ffill()
df_seq["BFill"] = df_seq["Value"].bfill()

display(df_seq)

,Time,Value,FFill,BFill
0,1,10.5,10.5,10.5
1,2,NaN,10.5,25.0
2,3,NaN,10.5,25.0
3,4,25.0,25.0,25.0
4,5,30.0,30.0,30.0


# 13. Interpolation

### Concept & Definition
Calculates missing values by fitting mathematical curves (linear, polynomial, spline) between adjacent known data points.

### When to Use:
- Smooth continuous numerical data ordered sequentially or temporally.

### When NOT to Use:
- Unordered tabular categorical or discrete data.

### Advantages & Limitations:
- **Advantage:** Smoother, more realistic approximations than mean/median imputation for sequential data.
- **Limitation:** Computationally more expensive and sensitive to surrounding noise.

In [13]:
# Linear Interpolation
df_interp = df_seq.copy()
df_interp["Linear_Interpolated"] = df_interp["Value"].interpolate(method="linear")

display(df_interp[["Value", "Linear_Interpolated"]])

,Value,Linear_Interpolated
0,10.5,10.500000
1,NaN,15.333333
2,NaN,20.166667
3,25.0,25.000000
4,30.0,30.000000


# 14. Group-Based Imputation

### Concept & Definition
Imputes missing values using aggregate statistics (mean, median) calculated within sub-groups of related categorical attributes.

### When to Use:
- When missing values have strong correlation with another complete feature (e.g., imputing `TotalCharges` based on `TenureYears` groups).

### When NOT to Use:
- When sample sizes within sub-groups are tiny (< 5 samples per group).

### Advantages & Limitations:
- **Advantage:** Higher domain accuracy than global mean/median.
- **Limitation:** Requires strong cross-feature relationships.

In [14]:
# Group-Based Imputation (Impute MonthlyCharges using median per TenureYears group)
df_group = df_detect.copy()

df_group["MonthlyCharges_GroupImputed"] = df_group.groupby("TenureYears")[
    "MonthlyCharges"
].transform(lambda group: group.fillna(group.median()))

print("Group-Imputed Missing Count:", df_group["MonthlyCharges_GroupImputed"].isnull().sum())

Group-Imputed Missing Count: 0


# 15. KNN (K-Nearest Neighbors) Imputation

### Concept & Definition
Uses Euclidean distance to find the $k$ most similar complete observations in multi-dimensional space and averages their values to fill missing entries.

### When to Use:
- Complex tabular datasets with strong multi-variable correlations (MAR mechanism).

### When NOT to Use:
- Massive multi-million row production pipelines (high computational complexity $\mathcal{O}(N^2)$).

### Advantages & Limitations:
- **Advantage:** Highly accurate multi-variable imputation.
- **Limitation:** Requires scaled numeric features and scales poorly with data size.

In [15]:
from sklearn.impute import KNNImputer

# Select numeric features for KNN
df_knn = df_detect[["Age", "MonthlyCharges", "TenureYears"]].copy()

imputer_knn = KNNImputer(n_neighbors=5)
df_knn_imputed = pd.DataFrame(
    imputer_knn.fit_transform(df_knn), columns=df_knn.columns
)

print("=== KNN Imputation Completed ===")
print("Null count after KNN:", df_knn_imputed.isnull().sum().to_dict())
display(df_knn_imputed.head(5))

=== KNN Imputation Completed ===
Null count after KNN: {'Age': 0, 'MonthlyCharges': 0, 'TenureYears': 0}


,Age,MonthlyCharges,TenureYears
0,34.0,29.85,3.0
1,150.0,56.95,10.0
2,52.0,105.50,2.0
3,45.0,56.95,10.0
4,25.0,29.85,3.0


# 16. Iterative Imputation (Multivariate Imputation by Chained Equations - MICE)

### Concept & Definition
Models each feature with missing values as a function of all other features in a round-robin sequential regression loop until convergence.

### When to Use:
- Advanced ML research pipelines with complex continuous data distributions.

### When NOT to Use:
- Latency-sensitive real-time deployment pipelines.

### Advantages & Limitations:
- **Advantage:** Gold-standard accuracy for multivariate missing data.
- **Limitation:** Computationally intensive and can overfit if improperly tuned.

In [16]:
# Iterative Imputer Implementation
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

df_mice = df_detect[["Age", "MonthlyCharges", "TenureYears"]].copy()

mice_imputer = IterativeImputer(max_iter=10, random_state=42)
df_mice_imputed = pd.DataFrame(
    mice_imputer.fit_transform(df_mice), columns=df_mice.columns
)

print("=== Iterative MICE Imputation Completed ===")
display(df_mice_imputed.head(5))

=== Iterative MICE Imputation Completed ===


,Age,MonthlyCharges,TenureYears
0,34.0,29.85,3.0
1,150.0,56.95,10.0
2,52.0,105.50,2.0
3,45.0,56.95,10.0
4,25.0,29.85,3.0


# 17. Summary & Best Practices for Missing Value Imputation

### Imputation Selection Strategy:
1. **Numerical (Skewed / Outliers):** Use **Median Imputation** or **KNN Imputation**.
2. **Numerical (Normal Distribution):** Use **Mean Imputation**.
3. **Categorical:** Use **Mode Imputation** or **Constant Imputation (`"Unknown"`)**.
4. **Time Series / Sequential:** Use **Forward/Backward Fill** or **Linear Interpolation**.
5. **High Missingness (> 50%):** Consider **Column Dropping**.

> **Golden Rule:** Never perform imputation on the complete dataset before splitting! Fit imputers *only* on the training set to prevent data leakage.

In [17]:
# Save clean imputed checkpoint for future notebooks
df_clean_imputed = df_detect.copy()

df_clean_imputed["Age"] = df_clean_imputed["Age"].fillna(
    df_clean_imputed["Age"].median()
)
df_clean_imputed["MonthlyCharges"] = df_clean_imputed["MonthlyCharges"].fillna(
    df_clean_imputed["MonthlyCharges"].median()
)
df_clean_imputed["Gender"] = df_clean_imputed["Gender"].fillna(
    df_clean_imputed["Gender"].mode()[0]
)

print("=== Final Checkpoint Missing Counts ===")
print(df_clean_imputed.isnull().sum())
print("\nNotebook 03 execution completed successfully!")

=== Final Checkpoint Missing Counts ===
CustomerID          0
Age                 0
Gender              0
TenureYears         0
MonthlyCharges      0
TotalCharges        0
ContractType      171
PaymentMethod       0
Churn               0
dtype: int64

Notebook 03 execution completed successfully!
